In [2]:
%pip install scikit-learn

  Using cached scikit_learn-1.9.1-cp313-cp313-macosx_12_0_arm64.whl.metadata (9.3 kB)
  Using cached scipy-1.18.1-cp313-cp313-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached threadpoolctl-3.8.0-py3-none-any.whl.metadata (29 kB)
Using cached scikit_learn-1.9.1-cp313-cp313-macosx_12_0_arm64.whl (8.3 MB)
Using cached joblib-1.6.0-py3-none-any.whl (306 kB)
Using cached scipy-1.18.1-cp313-cp313-macosx_14_0_arm64.whl (20.5 MB)
Using cached threadpoolctl-3.8.0-py3-none-any.whl (32 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [scikit-learn] [scikit-learn]

[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
# Imports used by preprocessing and the recommendation model.
import ast
import html
import re
import unicodedata
from pathlib import Path

import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MultiLabelBinarizer

In [4]:
# Load the source data and keep products with both feature sources available.
DATA_DIR = Path("data")
df = pd.read_csv(DATA_DIR / "product_info.csv")

products = (
    df.dropna(subset=["ingredients"])
    .copy()
    .reset_index(drop=True)
)

In [5]:
# convert str of list into python list
def parse_highlights(value):
    
    
    if isinstance(value, list):
        return value

    if pd.isna(value):
        return []

    try:
        parsed = ast.literal_eval(value)
        if isinstance(parsed, list):
            return parsed
    except (ValueError, SyntaxError):
        pass

    return [str(value)]


products["highlights_list"] = products["highlights"].apply(
    parse_highlights
)

In [6]:
def clean_highlight(text):
    # convert to lowercase
    text = str(text).lower().strip()

    # normalize unicode characters
    text = unicodedata.normalize("NFKC", text)

    # replace hyphens with spaces
    text = re.sub(r"[-‐‑–—]", " ", text)

    # remove extra spaces
    text = re.sub(r"\s+", " ", text)

    return text

products["cleaned_highlights"] = (
    products["highlights_list"].apply(
        lambda items: [clean_highlight(h) for h in items]
    )
)

In [7]:
#  now splite "good for: A,B,C" into "good for A, good for B ..."
skin_type_mapping = {
    "best for dry skin": ["skin_dry"],
    "best for oily skin": ["skin_oily"],
    "best for normal skin": ["skin_normal"],
    "best for combination skin": ["skin_combination"],
    "best for dry, combo, normal skin": [
        "skin_dry", "skin_combination", "skin_normal"
    ],
    "best for oily, combo, normal skin": [
        "skin_oily", "skin_combination", "skin_normal"
    ]
}

#  revomoving hihglights non-related
marketing_highlights = {
    "community favorite",
    "clean at sephora",
    "clean + planet positive",
    "black owned at sephora"
}

def normalize_highlight(highlight):
    # Normalize text formatting
    h = highlight.lower().strip()
    h = h.replace("-", " ")
    h = re.sub(r"\s+", " ", h)

    # Remove marketing features from similarity
    if h in marketing_highlights or h.startswith("allure "):
        return []

    # Expand combined skin-type labels
    if h in skin_type_mapping:
        return skin_type_mapping[h]

    # Normalize remaining highlights
    return [h.replace(" ", "_")]


    
def preprocess_highlights(highlights):
    normalized = []

    for highlight in highlights:
        normalized.extend(normalize_highlight(highlight))

    # Remove duplicates while preserving order
    return list(dict.fromkeys(normalized))


products["normalized_highlights"] = (
    products["cleaned_highlights"].apply(preprocess_highlights)
)

In [9]:
def normalize_ingredient(name):
    
    # clean some formatiing 
    if not isinstance(name, str):
        return ""

    #  decode some html values
    name = html.unescape(name)
    name = unicodedata.normalize("NFKC", name).casefold()

    # replace all possible dashes with one type
    name = re.sub(r"[‐‑‒–—−]", "-", name)

    # remove extra white spaces
    name = re.sub(r"\s+", " ", name).strip()
    name = re.sub(r"\s*/\s*", "/", name)

    name = re.sub(r"[\s*†‡]+$", "", name)
    # removes speces, periods, semicolms
    return name.strip(" .;")

# splite the ingridients, while being mindful of the brakets in notation
# "Water, Example Extract (leaf, stem), Glycerin" -> ["water", "example extract (leaf, stem)", "glycerin"]
# Making it numbers safe so we dont acesently splite. anumber into differnt ingridients
def split_ingredient_text(text):
    #  to store the names of ingridients
    ingredients = []
    
    # the current chars of the ingridient
    current = []
    
    # opened parathethis 
    stack = []

    # 1) find numeric prefixes such as "1,2-" or "1,2,3-".
    # 2) save the positions of their commas so they are not treated
    # as separators between ingredients.
    numeric_commas = set()

    numeric_prefix_pattern = r"(?<!\w)\d+(?:,\d+)+\s*[-‐‑‒–—−]"

    for match in re.finditer(numeric_prefix_pattern, text):
        for position in range(match.start(), match.end()):
            if text[position] == ",":
                numeric_commas.add(position)

    # read the ingredient text one character at a time.
    for position, char in enumerate(text):

        if char in "([":
            stack.append(char)

        elif char in ")]":
            expected = "(" if char == ")" else "["

            if not stack or stack[-1] != expected:
                raise ValueError(
                    "Unbalanced ingredient parentheses/brackets"
                )

            stack.pop()

        # Aa comma separates ingredients only if it is:
        # 1)outside parentheses and brackets and
        # 2)not part of a numeric prefix
        is_separator = (
            char == ","
            and not stack
            and position not in numeric_commas
        )

        if is_separator:
            ingredient = normalize_ingredient(
                "".join(current)
            )

            if ingredient:
                ingredients.append(ingredient)

            # starting collecting the next ingredient.
            current = []

        else:
            current.append(char)

    # An opening parenthesis or bracket was never closed.
    if stack:
        raise ValueError(
            "Unbalanced ingredient parentheses/brackets"
        )

    # Add the final ingredient, which has no comma after it.
    ingredient = normalize_ingredient(
        "".join(current)
    )

    if ingredient:
        ingredients.append(ingredient)

    return ingredients


# also assignes a stustues to an ingridient list
# missing -> no values
# invalid-list - > invalid python list
# invalid section - contains elem that is not a sring
# needs_section_review - contains more than one section of ingridients, like seperate part ususlly devided by : 
# needs_section_review - missing brakets
# empty
def parse_ingredients(value):
    if isinstance(value, list):
        sections = value

    elif isinstance(value, str):
        value = value.strip()

        if not value:
            return {"ingredients": [], "status": "missing"}

        if value.startswith("["):
            try:
                # converts text into actual list python
                sections = ast.literal_eval(value)
            except (ValueError, SyntaxError):
                return {"ingredients": [], "status": "invalid_list"}
        else:
            sections = [value]

    else:
        return {"ingredients": [], "status": "missing"}

    if not isinstance(sections, list):
        return {"ingredients": [], "status": "invalid_list"}

    if not all(isinstance(section, str) for section in sections):
        return {"ingredients": [], "status": "invalid_section"}

    sections = [section.strip() for section in sections if section.strip()]

    if not sections:
        return {"ingredients": [], "status": "missing"}

    if len(sections) > 1:
        return {"ingredients": [], "status": "needs_section_review"}

    try:
        ingredients = split_ingredient_text(sections[0])
    except ValueError:
        return {"ingredients": [], "status": "needs_delimiter_review"}

    return {
        "ingredients": ingredients,
        "status": "parsed" if ingredients else "empty",
    }

In [10]:
print(split_ingredient_text(
    "Water, 1,2-Hexanediol, Glycerin"
))

['water', '1,2-hexanediol', 'glycerin']


In [11]:
APPROVED_CORRECTIONS = {
    "hydroxyacetophe-none": "hydroxyacetophenone",
    "pan-thenol": "panthenol",
}

APPROVED_ALIASES = {
    "aqua/water/eau": "water",
    "water/aqua/eau": "water",
    "aqua (water)": "water",
    "water (aqua)": "water",
}

CANONICAL_MAP = {
    normalize_ingredient(source): normalize_ingredient(target)
    for source, target in {
        **APPROVED_CORRECTIONS,
        **APPROVED_ALIASES,
    }.items()
}


def canonicalize_ingredients(ingredients):
    # applying the approved replacement
    cleaned = []

    for ingredient in ingredients:
        name = normalize_ingredient(ingredient)
        name = CANONICAL_MAP.get(name, name)

        if name:
            cleaned.append(name)

    return list(dict.fromkeys(cleaned))

In [12]:
parsed = pd.DataFrame(
    products["ingredients"].apply(parse_ingredients).tolist(),
    index=products.index,
)

products["ingredients_list"] = parsed["ingredients"]
products["ingredient_parse_status"] = parsed["status"]


products["normalized_ingredients"] = (
    products["ingredients_list"].apply(canonicalize_ingredients)
)

print(products["ingredient_parse_status"].value_counts())

# recreate skincare after updating products.
skincare = products.loc[
    products["primary_category"].eq("Skincare")
].copy()

<unknown>:1: SyntaxWarning: invalid escape sequence '\A'
<unknown>:1: SyntaxWarning: invalid escape sequence '\A'
<unknown>:1: SyntaxWarning: invalid escape sequence '\A'


ingredient_parse_status
parsed                    6497
needs_section_review      1009
needs_delimiter_review      43
Name: count, dtype: int64


In [13]:
# want to find cheaper per ouces, for the better comparison will conver everyhting to ml
import re
import numpy as np
import pandas as pd


# get product size
def parse_product_size(size):
    if not isinstance(size, str):
        return np.nan, None

    size = size.lower().strip()

    # Reject multipacks that need special handling
    if re.search(r'\d+\s*[x×]\s*\d', size):
        return np.nan, None

    # Prefer explicitly stated metric units
    matches = re.findall(
        r'(?<!\w)(\d*\.?\d+)\s*(ml|milliliters?|grams?|g)\b',
        size
    )

    if matches:
        quantities = [
            (
                float(value),
                "mL" if unit.startswith(("ml", "milliliter")) else "g"
            )
            for value, unit in matches
        ]

        # Avoid choosing a size when multiple quantities conflict
        if len(set(quantities)) != 1:
            return np.nan, None

        value, unit = quantities[0]
        return (value, unit) if value > 0 else (np.nan, None)

    # Convert explicitly labeled fluid ounces to milliliters
    match = re.search(
        r'(?<!\w)(\d*\.?\d+)\s*fl\.?\s*oz\b',
        size
    )
    # Handle oz without an explicit fluid/weight designation
    match = re.search(
        r'(?<!\w)(\d*\.?\d+)\s*oz\b',
        size
    )

    if match:
        value = float(match.group(1))
        return value, "oz"

    if match:
        value_ml = float(match.group(1)) * 29.5735
        return (value_ml, "mL") if value_ml > 0 else (np.nan, None)

    # Ambiguous ounces, sets, or unknown units
    return np.nan, None

In [14]:
# Keep individual skincare formulas.
eligible_categories = {
    "Cleansers",
    "Eye Care",
    "Lip Balms & Treatments",
    "Masks",
    "Moisturizers",
    "Self Tanners",
    "Sunscreen",
    "Treatments",
}


# working only with skincare from chosen ligible_categories and parsed, so dont deal with complicated stuff
skincare_model = products.loc[
    products["primary_category"].eq("Skincare")
    & products["secondary_category"].isin(eligible_categories)
    & products["ingredient_parse_status"].eq("parsed")
    & products["normalized_ingredients"].map(bool)
].copy()

skincare_model["price_usd"] = pd.to_numeric(
    skincare_model["price_usd"],
    errors="coerce",
)

skincare_model = (
    skincare_model
    .drop_duplicates(subset="product_id")
    .reset_index(drop=True)
)

if skincare_model.empty:
    raise ValueError("No eligible products. Check the preprocessing cells.")

print("Products available:", len(skincare_model))

Products available: 1821


In [15]:
# Extract size and unit
skincare_model[["size_value", "size_unit"]] = pd.DataFrame(
    skincare_model["size"].apply(parse_product_size).tolist(),
    index=skincare_model.index,
)

# Calculate unit price
skincare_model["price_per_unit"] = (
    skincare_model["price_usd"] / skincare_model["size_value"]
)

# Inspect the results
display(
    skincare_model[
        [
            "product_name",
            "size",
            "size_value",
            "size_unit",
            "price_usd",
            "price_per_unit",
        ]
    ].head(20)
)

,product_name,size,size_value,size_unit,price_usd,price_per_unit
0,GENIUS Sleeping Collagen Moisturizer,2 oz/ 60 mL,60.0,mL,98.0,1.633333
1,GENIUS Liquid Collagen Serum,1 oz / 30 mL,30.0,mL,115.0,3.833333
2,GENIUS Liquid Collagen Lip Treatment,.5 oz / 15 mL,15.0,mL,29.0,1.933333
3,SUBLIME DEFENSE Ultra Lightweight UV Defense F...,1 oz,1.0,oz,28.0,28.000000
4,GENIUS Ultimate Anti-Aging Cream,2 oz/ 60 mL,60.0,mL,112.0,1.866667
5,GENIUS Ultimate Anti-Aging Melting Cleanser,5 oz/ 150 mL,150.0,mL,38.0,0.253333
6,Gentle Rejuvenating Cleanser,4 oz/ 120 mL,120.0,mL,28.0,0.233333
7,Advanced Anti-Aging Repairing Oil,1 oz/ 30 mL,30.0,mL,82.0,2.733333
8,Overnight Restorative Cream,2 oz/ 60 mL,60.0,mL,94.0,1.566667
9,GENIUS Ultimate Anti-Aging Vitamin C+ Serum,1 oz/ 30 mL,30.0,mL,118.0,3.933333


In [16]:
import numpy as np
from sklearn.preprocessing import normalize


# make rows aligned with both matrices.
skincare_model = skincare_model.reset_index(drop=True)


In [ ]:
%pip install transformers torch

In [ ]:
# incorporating textures from the reviews 
import re

import numpy as np
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
)



In [26]:
# just some models imports and set up

MODEL_NAME = "facebook/bart-large-mnli"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

nli_model = (
    AutoModelForSequenceClassification
    .from_pretrained(MODEL_NAME)
)

# will use a GPU if CUDA is available, otherwise use the CPU
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

nli_model = nli_model.to(device)
nli_model.eval()

print("Using device:", device)
print("Model labels:", nli_model.config.id2label)

W1128 11:05:22.386000 67524 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 515/515 [00:00<00:00, 7658.37it/s]

Using device: cpu
Model labels: {0: 'contradiction', 1: 'neutral', 2: 'entailment'}


In [38]:
TEXTURE_STATEMENTS = {
    "greasy": "The product feels greasy.",
    "sticky": "The product feels sticky.",
    "heavy": "The product feels heavy on the skin.",
    "lightweight": "The product feels lightweight.",
    "watery": "The product has a watery texture.",
    "creamy": "The product has a creamy texture.",
    
    "gel_like": "The product has a gel-like texture.",
    
    "thick": "The product has a thick consistency.",

    "thin": "The product has a thin consistency.",

    "silky": "The product feels silky on the skin.",

    "smooth": "The product feels smooth on the skin.",

    "fast_absorbing": "The product absorbs quickly into the skin.",

    "dewy": "The product leaves a dewy finish.",

    "matte": "The product leaves a matte finish.",

    "pilling": "The product pills or rolls up on the skin.",

    "grainy": "The product has a grainy texture.",

    "drying": "The product makes the skin feel dry.",
}


TEXTURE_CUES = {
    "greasy": r"\b(?:greasy|oily|slick|oiliness)\b",
    "sticky": r"\b(?:sticky|tacky|stickiness)\b",
    "heavy": r"\b(?:heavy|thick|dense|rich)\b",
    "lightweight": r"\b(?:lightweight|light|weightless)\b",
    "watery": r"\b(?:watery|runny|liquid)\b",
    "creamy": r"\b(?:creamy|creaminess|buttery)\b",
    "gel_like": (
        r"\b(?:gel|gel-like|gelly|jelly)\b"
    ),

    "thick": (
        r"\b(?:thick|thickness|dense)\b"
    ),

    "thin": (
        r"\b(?:thin|thinness|runny|fluid)\b"
    ),

    "silky": (
        r"\b(?:silky|silk-like|velvety|velvet-like)\b"
    ),

    "smooth": (
        r"\b(?:smooth|smoothly|smoothness)\b"
    ),

    "fast_absorbing": (
        r"\b(?:absorb|absorbs|absorbed|absorbing)\b"
        r"|\bsinks?\s+(?:in|into)\b"
        r"|\bsoaks?\s+in\b"
    ),

    "dewy": (
        r"\b(?:dewy|glowy|glossy)\b"
    ),

    "matte": (
        r"\b(?:matte|mattifying|matte-looking)\b"
    ),

    "pilling": (
        r"\b(?:pill|pills|pilled|pilling)\b"
        r"|\bballs?\s+up\b"
        r"|\brolls?\s+up\b"
    ),

    "grainy": (
        r"\b(?:grainy|gritty|grittiness)\b"
    ),

    "drying": (
        r"\b(?:drying|dried|dry|dryness)\b"
    ),
}

In [39]:
def get_label_indexes(model):
    labels = {
        str(label).lower(): int(index)
        for index, label in model.config.id2label.items()
    }

    required = {
        "entailment",
        "contradiction",
        "neutral",
    }
    
    if not required.issubset(labels):
        raise ValueError(
            f"Unexpected model labels: {labels}"
        )

    return labels

LABEL_INDEXES = get_label_indexes(nli_model)

print(LABEL_INDEXES)

{'contradiction': 0, 'neutral': 1, 'entailment': 2}


In [40]:
def split_review_sentences(review_text):
    # splittign reviews into sentances, or new line seperated parts
    
    if not isinstance(review_text, str):
        return []

    review_text = review_text.strip()

    if not review_text:
        return []

    passages = re.split(
        r"(?<=[.!?])\s+|\n+",
        review_text,
    )

    return [
        passage.strip()
        for passage in passages
        if passage.strip()
    ]

In [41]:
"""
Analyzing each review sentence against each texture statement.
threshold is a starting value that needs validation.

Returns:
    - a table of sentence-level model scores
    - review-level flags for supported and contradicted traits
"""
def classify_texture(review_text, threshold=0.65):
    sentences = split_review_sentences(review_text)
    
    if not sentences:
        return pd.DataFrame(), {}
    
    
     # build one sentence-statement pair for every combination
    premises = []
    hypotheses = []
    pair_details = []
    
    
    for sentence in sentences:
        for texture_name, statement in TEXTURE_STATEMENTS.items():
            
            
            # only ask about this texture if the sentence
            # contains one of its candidate words
            cue_found = re.search(
                TEXTURE_CUES[texture_name],
                sentence,
                flags=re.IGNORECASE,
            )

            if not cue_found:
                continue

            premises.append(sentence)
            hypotheses.append(statement)

            pair_details.append({
                "sentence": sentence,
                "texture": texture_name,
                "statement": statement,
            })
            
            
            
    # no texture cues were found anywhere in the review
    if not premises:
        empty_features = {}

        for texture_name in TEXTURE_STATEMENTS:
            empty_features[f"{texture_name}_present"] = 0
            empty_features[f"{texture_name}_negated"] = 0

        return pd.DataFrame(), empty_features



    # tokenixer call 
    encoded = tokenizer(
    premises,
    hypotheses,
    padding=True,
    truncation="only_first",
    max_length=512,
    return_tensors="pt",
    )

    encoded = {
        name: tensor.to(device)
        for name, tensor in encoded.items()
    }
    
    
    # Make predictions without calculating gradients.
    with torch.inference_mode():
        logits = nli_model(**encoded).logits

        probabilities = torch.softmax(
            logits,
            dim=1,
        ).cpu().numpy()


    rows = []
    
    
    for index, details in enumerate(pair_details):

        entailment = probabilities[
            index,
            LABEL_INDEXES["entailment"],
        ]

        contradiction = probabilities[
            index,
            LABEL_INDEXES["contradiction"],
        ]

        neutral = probabilities[
            index,
            LABEL_INDEXES["neutral"],
        ]
        
        
        # If the model is not sufficiently confident,
        # keep the result as unclear.
        if (
            entailment >= threshold
            and entailment > contradiction
            and entailment > neutral
        ):
            prediction = "present"

        elif (
            contradiction >= threshold
            and contradiction > entailment
            and contradiction > neutral
        ):
            prediction = "negated"

        else:
            prediction = "unclear"

        rows.append({
            **details,
            "entailment": float(entailment),
            "contradiction": float(contradiction),
            "neutral": float(neutral),
            "prediction": prediction,
        })
        
        sentence_results = pd.DataFrame(rows)

        # A review may contain both positive and negated
        # mentions in different sentences.
        review_features = {}
        
        
        for texture_name in TEXTURE_STATEMENTS:

            texture_rows = sentence_results.loc[
                sentence_results["texture"].eq(texture_name)
            ]

            review_features[f"{texture_name}_present"] = int(
                texture_rows["prediction"].eq("present").any()
            )

            review_features[f"{texture_name}_negated"] = int(
                texture_rows["prediction"].eq("negated").any()
            )

    return sentence_results, review_features

    
    

In [42]:
test_reviews = [
    "I did not like the packaging. The cream feels sticky.",

    "I have oily skin, but this moisturizer does not feel greasy.",

    "It is not sticky at first. It becomes sticky later.",

    "The packaging is beautiful.",
    "The moisturizer has a thick texture but feels lightweight.",

    "It sinks into my skin quickly and leaves a dewy finish.",

    "The serum does not pill under sunscreen.",

    "I have dry skin, but this cream does not dry it out.",

    "The formula feels silky and smooth.",
]

for review in test_reviews:

    sentence_results, review_features = classify_texture(
        review
    )

    print("\nREVIEW:")
    print(review)

    print("\nREVIEW FEATURES:")
    print(review_features)

    print("\nSENTENCE-LEVEL PREDICTIONS:")

    # Put the new check here, inside the loop.
    if sentence_results.empty:
        print("No texture cues found.")

    else:
        display(
            sentence_results.loc[
                sentence_results["prediction"].ne("unclear"),
                [
                    "sentence",
                    "texture",
                    "prediction",
                    "entailment",
                    "contradiction",
                    "neutral",
                ],
            ]
        )


REVIEW:
I did not like the packaging. The cream feels sticky.

REVIEW FEATURES:
{'greasy_present': 0, 'greasy_negated': 0, 'sticky_present': 1, 'sticky_negated': 0, 'heavy_present': 0, 'heavy_negated': 0, 'lightweight_present': 0, 'lightweight_negated': 0, 'watery_present': 0, 'watery_negated': 0, 'creamy_present': 0, 'creamy_negated': 0, 'gel_like_present': 0, 'gel_like_negated': 0, 'thick_present': 0, 'thick_negated': 0, 'thin_present': 0, 'thin_negated': 0, 'silky_present': 0, 'silky_negated': 0, 'smooth_present': 0, 'smooth_negated': 0, 'fast_absorbing_present': 0, 'fast_absorbing_negated': 0, 'dewy_present': 0, 'dewy_negated': 0, 'matte_present': 0, 'matte_negated': 0, 'pilling_present': 0, 'pilling_negated': 0, 'grainy_present': 0, 'grainy_negated': 0, 'drying_present': 0, 'drying_negated': 0}

SENTENCE-LEVEL PREDICTIONS:


,sentence,texture,prediction,entailment,contradiction,neutral
0,The cream feels sticky.,sticky,present,0.986962,0.000604,0.012435



REVIEW:
I have oily skin, but this moisturizer does not feel greasy.

REVIEW FEATURES:
{'greasy_present': 0, 'greasy_negated': 1, 'sticky_present': 0, 'sticky_negated': 0, 'heavy_present': 0, 'heavy_negated': 0, 'lightweight_present': 0, 'lightweight_negated': 0, 'watery_present': 0, 'watery_negated': 0, 'creamy_present': 0, 'creamy_negated': 0, 'gel_like_present': 0, 'gel_like_negated': 0, 'thick_present': 0, 'thick_negated': 0, 'thin_present': 0, 'thin_negated': 0, 'silky_present': 0, 'silky_negated': 0, 'smooth_present': 0, 'smooth_negated': 0, 'fast_absorbing_present': 0, 'fast_absorbing_negated': 0, 'dewy_present': 0, 'dewy_negated': 0, 'matte_present': 0, 'matte_negated': 0, 'pilling_present': 0, 'pilling_negated': 0, 'grainy_present': 0, 'grainy_negated': 0, 'drying_present': 0, 'drying_negated': 0}

SENTENCE-LEVEL PREDICTIONS:


,sentence,texture,prediction,entailment,contradiction,neutral
0,"I have oily skin, but this moisturizer does no...",greasy,negated,0.002434,0.996571,0.000995



REVIEW:
It is not sticky at first. It becomes sticky later.

REVIEW FEATURES:
{'greasy_present': 0, 'greasy_negated': 0, 'sticky_present': 1, 'sticky_negated': 1, 'heavy_present': 0, 'heavy_negated': 0, 'lightweight_present': 0, 'lightweight_negated': 0, 'watery_present': 0, 'watery_negated': 0, 'creamy_present': 0, 'creamy_negated': 0, 'gel_like_present': 0, 'gel_like_negated': 0, 'thick_present': 0, 'thick_negated': 0, 'thin_present': 0, 'thin_negated': 0, 'silky_present': 0, 'silky_negated': 0, 'smooth_present': 0, 'smooth_negated': 0, 'fast_absorbing_present': 0, 'fast_absorbing_negated': 0, 'dewy_present': 0, 'dewy_negated': 0, 'matte_present': 0, 'matte_negated': 0, 'pilling_present': 0, 'pilling_negated': 0, 'grainy_present': 0, 'grainy_negated': 0, 'drying_present': 0, 'drying_negated': 0}

SENTENCE-LEVEL PREDICTIONS:


,sentence,texture,prediction,entailment,contradiction,neutral
0,It is not sticky at first.,sticky,negated,0.019662,0.966503,0.013835
1,It becomes sticky later.,sticky,present,0.860390,0.006340,0.133270


... (6 outputs left out)


REVIEW:
I have dry skin, but this cream does not dry it out.

REVIEW FEATURES:
{'greasy_present': 0, 'greasy_negated': 0, 'sticky_present': 0, 'sticky_negated': 0, 'heavy_present': 0, 'heavy_negated': 0, 'lightweight_present': 0, 'lightweight_negated': 0, 'watery_present': 0, 'watery_negated': 0, 'creamy_present': 0, 'creamy_negated': 0, 'gel_like_present': 0, 'gel_like_negated': 0, 'thick_present': 0, 'thick_negated': 0, 'thin_present': 0, 'thin_negated': 0, 'silky_present': 0, 'silky_negated': 0, 'smooth_present': 0, 'smooth_negated': 0, 'fast_absorbing_present': 0, 'fast_absorbing_negated': 0, 'dewy_present': 0, 'dewy_negated': 0, 'matte_present': 0, 'matte_negated': 0, 'pilling_present': 0, 'pilling_negated': 0, 'grainy_present': 0, 'grainy_negated': 0, 'drying_present': 0, 'drying_negated': 1}

SENTENCE-LEVEL PREDICTIONS:


,sentence,texture,prediction,entailment,contradiction,neutral
0,"I have dry skin, but this cream does not dry i...",drying,negated,0.001127,0.99784,0.001033



REVIEW:
The formula feels silky and smooth.

REVIEW FEATURES:
{'greasy_present': 0, 'greasy_negated': 0, 'sticky_present': 0, 'sticky_negated': 0, 'heavy_present': 0, 'heavy_negated': 0, 'lightweight_present': 0, 'lightweight_negated': 0, 'watery_present': 0, 'watery_negated': 0, 'creamy_present': 0, 'creamy_negated': 0, 'gel_like_present': 0, 'gel_like_negated': 0, 'thick_present': 0, 'thick_negated': 0, 'thin_present': 0, 'thin_negated': 0, 'silky_present': 1, 'silky_negated': 0, 'smooth_present': 1, 'smooth_negated': 0, 'fast_absorbing_present': 0, 'fast_absorbing_negated': 0, 'dewy_present': 0, 'dewy_negated': 0, 'matte_present': 0, 'matte_negated': 0, 'pilling_present': 0, 'pilling_negated': 0, 'grainy_present': 0, 'grainy_negated': 0, 'drying_present': 0, 'drying_negated': 0}

SENTENCE-LEVEL PREDICTIONS:


,sentence,texture,prediction,entailment,contradiction,neutral
0,The formula feels silky and smooth.,silky,present,0.859786,0.001123,0.139091
1,The formula feels silky and smooth.,smooth,present,0.946306,0.000349,0.053345


In [43]:


# initialised the Tf-idf using ingridients
ingredient_vectorizer = TfidfVectorizer(
    analyzer=lambda ingredients: ingredients,
    lowercase=False,
)

# actully apply the vectorised and assign frequencies to the ingridients
X_ingredients = ingredient_vectorizer.fit_transform(
    skincare_model["normalized_ingredients"]
)

# we want our ingridient list to be sensitive to the positioning/order
# decay - how fast the weight destreas, the greatwe -> slower decrease
def position_weight(position, decay=8, min_weight=0.3):
    return min_weight + (1 - min_weight) * np.exp(
        -position / decay
    )

# Preserve original TF-IDF matrix
X_ingredients_original = X_ingredients.copy()

# Create position-weighted TF-IDF matrix
X_ingredients_position = X_ingredients.copy().tolil()

# Map ingredient name to its TF-IDF column
vocab = ingredient_vectorizer.vocabulary_

for row_idx, ingredients in enumerate(
    skincare_model["normalized_ingredients"]
):
    for position, ingredient in enumerate(ingredients):

        col_idx = vocab[ingredient]

        weight = position_weight(position)

        # now actully adjusting the TF-IDF score based on position
        X_ingredients_position[row_idx, col_idx] *= weight

# Convert back to efficient sparse matrix format
X_ingredients_position = normalize(
    X_ingredients_position.tocsr(),
    norm="l2"
)

print("Position-weighted matrix:", X_ingredients_position.shape)




# Now highlights
# each highlight becomes a binary feature: present or absent
highlight_encoder = MultiLabelBinarizer(sparse_output=True)

X_highlights = highlight_encoder.fit_transform(
    skincare_model["normalized_highlights"]
)


# makes a dictionary products id: row number
product_positions = {
    product_id: position
    for position, product_id in enumerate(skincare_model["product_id"])
}

print("Ingredient feature matrix:", X_ingredients.shape)
print("Highlight feature matrix:", X_highlights.shape)

Position-weighted matrix: (1821, 5658)
Ingredient feature matrix: (1821, 5658)
Highlight feature matrix: (1821, 67)


In [44]:
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd

In [45]:
# minimum number of nonempty reviews a product must have
MIN_REVIEWS = 10

# maximum number of reviews we will send to the model for each product
MAX_REVIEWS_PER_PRODUCT = 20


# for the first test, process only 10 eligible products
# will change this to None after checking accuracy and runtime
MAX_PRODUCTS_TO_PROCESS = 10

rng = np.random.default_rng(42)

model_ids = set(skincare_model["product_id"])


# count all nonempty review rows for each modeled product
review_counts = defaultdict(int)
review_samples = defaultdict(list)

review_paths = sorted(
    DATA_DIR.glob("reviews_*.csv")
)

if not review_paths:
    raise FileNotFoundError(
        "No review files found in the data folder."
    )

for path in review_paths:

    for chunk in pd.read_csv(
        path,
        usecols=["product_id", "review_text"],
        chunksize=100_000,
        low_memory=False,
    ):

        chunk = chunk.loc[
            chunk["product_id"].isin(model_ids)
        ].dropna(subset=["review_text"])

        for product_id, review_text in chunk[
            ["product_id", "review_text"]
        ].itertuples(index=False, name=None):

            if not isinstance(review_text, str):
                continue

            review_text = review_text.strip()

            if not review_text:
                continue

            review_counts[product_id] += 1
            count = review_counts[product_id]

            samples = review_samples[product_id]

            if len(samples) < MAX_REVIEWS_PER_PRODUCT:
                samples.append(review_text)

            else:
                
                replacement_index = rng.integers(0, count)

                if replacement_index < MAX_REVIEWS_PER_PRODUCT:
                    samples[replacement_index] = review_text

In [46]:
# selecting producs with enough reviews

eligible_review_ids = sorted([
    product_id
    for product_id, count in review_counts.items()
    if count >= MIN_REVIEWS
])

print(
    "Products with at least 10 nonempty reviews:",
    len(eligible_review_ids),
)

if MAX_PRODUCTS_TO_PROCESS is None:
    products_to_process = eligible_review_ids

else:
    sample_size = min(
        MAX_PRODUCTS_TO_PROCESS,
        len(eligible_review_ids),
    )

    products_to_process = rng.choice(
        eligible_review_ids,
        size=sample_size,
        replace=False,
    ).tolist()

print(
    "Products selected for this model run:",
    len(products_to_process),
)

Products with at least 10 nonempty reviews: 1672
Products selected for this model run: 10


In [47]:
texture_feature_names = []

for texture in TEXTURE_STATEMENTS:
    texture_feature_names.append(
        f"{texture}_present"
    )

    texture_feature_names.append(
        f"{texture}_negated"
    )

In [48]:
texture_rows = []

for product_number, product_id in enumerate(
    products_to_process,
    start=1,
):
    sampled_reviews = review_samples[product_id]

    # keep one prediction dictionary per review.
    review_feature_rows = []

    for review_text in sampled_reviews:
        _, review_features = classify_texture(
            review_text
        )

        review_feature_rows.append(review_features)

    # one row per sampled review.
    #
    # a review with no detected cue has zeros, it means
    # no prediction was recorded, not that the texture
    # is absent from the product
    feature_table = (
        pd.DataFrame(review_feature_rows)
        .reindex(columns=texture_feature_names)
        .fillna(0)
    )

    # average each feature over the sampled reviews
    mean_features = feature_table.mean()

    row = {
        "product_id": product_id,
        "total_nonempty_reviews": review_counts[product_id],
        "reviews_analyzed": len(sampled_reviews),
    }

    for feature in texture_feature_names:
        row[feature] = mean_features[feature]

    texture_rows.append(row)

    print(
        f"Processed {product_number}/"
        f"{len(products_to_process)} products"
    )

Processed 1/10 products
Processed 2/10 products
Processed 3/10 products
Processed 4/10 products
Processed 5/10 products
Processed 6/10 products
Processed 7/10 products
Processed 8/10 products
Processed 9/10 products
Processed 10/10 products


In [49]:
texture_profiles = pd.DataFrame(
    texture_rows,
    columns=[
        "product_id",
        "total_nonempty_reviews",
        "reviews_analyzed",
        *texture_feature_names,
    ],
)

display(texture_profiles.head())

,product_id,total_nonempty_reviews,reviews_analyzed,greasy_present,greasy_negated,sticky_present,sticky_negated,heavy_present,heavy_negated,lightweight_present,...,dewy_present,dewy_negated,matte_present,matte_negated,pilling_present,pilling_negated,grainy_present,grainy_negated,drying_present,drying_negated
0,P503928,27,20,0.000000,0.100000,0.0,0.00,0.0,0.0,0.05,...,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.15,0.150000
1,P450262,443,20,0.000000,0.050000,0.0,0.00,0.0,0.0,0.00,...,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.150000
2,P461199,22,20,0.000000,0.100000,0.0,0.05,0.0,0.0,0.05,...,0.05,0.05,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.150000
3,P417359,14,14,0.000000,0.000000,0.0,0.00,0.0,0.0,0.00,...,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000
4,P504919,19,19,0.210526,0.157895,0.0,0.00,0.0,0.0,0.00,...,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.052632


In [50]:
from scipy.sparse import csr_matrix

# aligning profiles with skincare_model's row order.
aligned_texture = (
    texture_profiles
    .set_index("product_id")
    .reindex(skincare_model["product_id"])
)

# products not processed, below the review threshold,
# or without a profile receive zero rows
texture_values = (
    aligned_texture[texture_feature_names]
    .fillna(0.0)
    .to_numpy(dtype=float)
)

X_texture = csr_matrix(texture_values)

# the model can use texture only for a product that was
# processed and has at least one detected texture feature
has_texture = (
    aligned_texture["reviews_analyzed"].notna().to_numpy()
    & (X_texture.getnnz(axis=1) > 0)
)

print("Texture matrix shape:", X_texture.shape)

print(
    "Products with usable texture vectors:",
    has_texture.sum(),
)

print(
    "Products using fallback:",
    (~has_texture).sum(),
)

assert X_texture.shape[0] == len(skincare_model)

Texture matrix shape: (1821, 34)
Products with usable texture vectors: 9
Products using fallback: 1812


In [51]:
print("Ingredients:", X_ingredients_position.shape)
print("Highlights: ", X_highlights.shape)
print("Texture:    ", X_texture.shape)

Ingredients: (1821, 5658)
Highlights:  (1821, 67)
Texture:     (1821, 34)


In [52]:
def recommend_dupes(
    product_id,
    top_k=5,
    ingredient_weight=0.5,
    texture_weight=0.3,
    highlight_weight=0.2,
    different_brand=False,
    position_importance=0.4,
):
    if product_id not in product_positions:
        raise ValueError("Product ID is not in skincare_model.")

    if not isinstance(top_k, int) or top_k < 1:
        raise ValueError("top_k must be a positive integer.")

    if not 0 < ingredient_weight < 1:
        raise ValueError("ingredient_weight must be between 0 and 1.")
    
    if ingredient_weight <= 0:
        raise ValueError(
            "ingredient_weight must be greater than 0."
        )

    if texture_weight < 0 or highlight_weight < 0:
        raise ValueError(
            "Texture and highlight weights cannot be negative."
        )

    position = product_positions[product_id]
    
    # what are we looking an alternative for
    query = skincare_model.iloc[position]


    # price validation usin gprice per unit
    if (pd.isna(query["price_per_unit"]) or not np.isfinite(query["price_per_unit"]) or query["price_per_unit"] <= 0):
        raise ValueError(
            "The selected product needs a valid price and size."
        )


    


    # original TF-IDF similarity (ignores ordering), using kust cosine
    original_scores = cosine_similarity(
        X_ingredients_original[position],
        X_ingredients_original,
    ).ravel()

    # position-aware TF-IDF similarity
    position_scores = cosine_similarity(
        X_ingredients_position[position],
        X_ingredients_position,
    ).ravel()

    # blend both measures, can be changed
    # position_importance defined in the fucntion call

    ingredient_scores = (
        (1 - position_importance) * original_scores
        + position_importance * position_scores
    )

    
    # want to make a highlight an optional,
    # so it is empty then we rely on ingridients only
    
    # 1 - ceck whether the query has at least one usable highlight
    query_has_highlights = X_highlights[position].nnz > 0
    # check if canditate has a highlight
    candidate_has_highlights = (
    X_highlights.getnnz(axis=1) > 0
    )
    # use highligh only when both have non empty
    use_highlights = (
    query_has_highlights & candidate_has_highlights
    )
    
   
    # ingridient score
    score_numerator = (
        ingredient_weight * ingredient_scores
    )
    
    score_denominator = np.full(
    len(skincare_model),
    ingredient_weight,
    dtype=float,
    )
    
    # adding texture only when both producst have it
    texture_scores = np.zeros(len(skincare_model))

    texture_used = np.zeros(
        len(skincare_model),
        dtype=bool,
    )

    if texture_weight > 0 and has_texture[position]:

        texture_scores = cosine_similarity(
            X_texture[position],
            X_texture,
        ).ravel()

        # the query has texture, select candidates that do too
        texture_used = has_texture.copy()

        score_numerator[texture_used] += (
            texture_weight
            * texture_scores[texture_used]
        )

        score_denominator[texture_used] += (
            texture_weight
        )
        
        
    # adding highlight only when both have them
    highlight_scores = np.zeros(len(skincare_model))

    highlights_used = np.zeros(
        len(skincare_model),
        dtype=bool,
    )

    query_has_highlights = (
        X_highlights[position].nnz > 0
    )

    candidate_has_highlights = (
        X_highlights.getnnz(axis=1) > 0
    )

    if (highlight_weight > 0 and X_highlights.shape[1] > 0
        and query_has_highlights):
        
        highlight_scores = cosine_similarity(
            X_highlights[position],
            X_highlights,
        ).ravel()

        highlights_used = candidate_has_highlights

        score_numerator[highlights_used] += (
            highlight_weight
            * highlight_scores[highlights_used]
        )

        score_denominator[highlights_used] += (
            highlight_weight
        )
    
    
    
   
    combined_scores = (
        score_numerator / score_denominator
    )
    
    
    
    

    # only looking at the comparable product categories
    mask = skincare_model["secondary_category"].eq(
        query["secondary_category"]
    )

    if pd.notna(query["tertiary_category"]):
        mask &= skincare_model["tertiary_category"].eq(
            query["tertiary_category"]
        )
    else:
        mask &= skincare_model["tertiary_category"].isna()

    # alternatives MUST have a lower package price
    # Compare only products with the same measurement unit
    mask &= skincare_model["size_unit"].eq(query["size_unit"])

    # Alternatives MUST have a lower price per unit
    mask &= (
        skincare_model["price_per_unit"].gt(0)
        & skincare_model["price_per_unit"].lt(query["price_per_unit"])
    )

    mask &= skincare_model["product_id"].ne(product_id)

    if different_brand:
        mask &= skincare_model["brand_name"].ne(query["brand_name"])

    # require some overlap in Only ingredients, highlights can be missing
    # tha same as highlishs applyies to texture
    mask &= (ingredient_scores > 0)


    results = skincare_model.loc[mask].copy()
    selected = mask.to_numpy()
    
    

    results["ingredient_similarity"] = ingredient_scores[selected]
    results["texture_similarity"] = np.where(texture_used[selected],texture_scores[selected],np.nan,)
    
    results["texture_used"] = (texture_used[selected])
    results["highlight_similarity"] = np.where(highlights_used[selected],highlight_scores[selected],np.nan,)
    results["highlights_used"] = (highlights_used[selected])
    
    results["combined_similarity"] = combined_scores[selected]

    results = results.sort_values(
        ["combined_similarity", "price_per_unit", "product_id"],
        ascending=[False, True, True],
    ).head(top_k).copy()

    # explain what each alternative shares with the selected product
    query_ingredients = set(query["normalized_ingredients"])
    query_highlights = set(query["normalized_highlights"])

    results["shared_ingredients"] = results[
        "normalized_ingredients"
    ].apply(lambda items: sorted(query_ingredients & set(items)))

    results["shared_highlights"] = results[
        "normalized_highlights"
    ].apply(lambda items: sorted(query_highlights & set(items)))

    results["package_savings_usd"] = (
        query["price_usd"] - results["price_usd"]
    )

    results["package_savings_pct"] = (
        100 * results["package_savings_usd"] / query["price_usd"]
    )
    
    results["unit_savings_usd"] = (
    query["price_per_unit"] - results["price_per_unit"])

    results["unit_savings_pct"] = (
        100 * results["unit_savings_usd"] / query["price_per_unit"]
    )
    
    

    columns = [
        "product_id",
        "product_name",
        "brand_name",
        "size",
        "price_usd",
        "ingredient_similarity",
        "highlight_similarity",
        "texture_similarity",
        "combined_similarity",
        "package_savings_usd",
        "package_savings_pct",
        "shared_highlights",
        "shared_ingredients",
        "highlights_used",
        "texture_used",
        
    ]

    return results[columns].reset_index(drop=True)

In [53]:
print(products_to_process[:5])

['P503928', 'P450262', 'P461199', 'P417359', 'P504919']


In [54]:
query_id = products_to_process[0]

display(
    skincare_model.loc[
        skincare_model["product_id"].eq(query_id),
        [
            "product_id",
            "product_name",
            "price_per_unit",
            "normalized_highlights",
        ],
    ]
)

,product_id,product_name,price_per_unit,normalized_highlights
776,P503928,Green Tea Hyaluronic Acid Face Cleanser,0.08,"[hyaluronic_acid, hydrating, without_mineral_o..."


In [55]:
recommendations = recommend_dupes(
    product_id=query_id,
    top_k=5,
)

display(recommendations)

,product_id,product_name,brand_name,size,price_usd,ingredient_similarity,highlight_similarity,texture_similarity,combined_similarity,package_savings_usd,package_savings_pct,shared_highlights,shared_ingredients,highlights_used,texture_used
0,P479353,Hyaluronic Acid Cleanser,The INKEY List,5 oz/ 150 mL,10.99,0.054258,0.530330,NaN,0.190278,1.01,8.416667,"[good_for:_dryness, hyaluronic_acid, hydrating]","[ethylhexylglycerin, glycerin, stearic acid, x...",True,False
1,P455364,Oat Cleansing Balm,The INKEY List,5 oz/ 150 mL,11.99,0.010165,0.408248,NaN,0.123903,0.01,0.083333,"[good_for:_dryness, skin_normal]","[1,2-hexanediol]",True,False
2,P469088,Fulvic Acid Brightening Cleanser,The INKEY List,5 oz/ 150 mL,11.99,0.029942,0.000000,NaN,0.021387,0.01,0.083333,[],"[butylene glycol, disodium edta, glycerin, xan...",True,False
3,P443833,Salicylic Acid Acne + Pore Cleanser,The INKEY List,5 oz/ 150 mL,11.99,0.027275,0.000000,NaN,0.019482,0.01,0.083333,[],"[ethylhexylglycerin, glycerin, propanediol, wa...",True,False
4,P471784,Gentle Foaming Cleanser,Mario Badescu,6 oz/ 177 mL,14.00,0.011531,NaN,NaN,0.011531,-2.00,-16.666667,[],"[ethylhexylglycerin, glycerin]",False,False


In [ ]:
# HERE put the product id you are looking dupes for 
# for example select the first product with usable and a valid price per unit
example_products = skincare_model.loc[
    skincare_model["price_per_unit"].gt(0)
    & np.isfinite(skincare_model["price_per_unit"])
]




if example_products.empty:
    raise ValueError("No products have a valid price per unit.")

# assign query_id to the product if you are looking a dupe for 
query_id = example_products.iloc[0]["product_id"]

display(
    skincare_model.loc[
        skincare_model["product_id"].eq(query_id),
        [
            "product_id", "product_name", "brand_name", "size",
            "price_usd", "normalized_highlights",
        ],
    ]
)

# Every returned alternative must be cheaper and share ingredients and highlights.
recommendations = recommend_dupes(
    product_id=query_id,
    top_k=5,
    ingredient_weight=0.7,
    different_brand=False,
)

display(
    recommendations.round({
        "ingredient_similarity": 3,
        "highlight_similarity": 3,
        "combined_similarity": 3,
        "package_savings_usd": 2,
        "package_savings_pct": 1,
    })
)

if recommendations.empty:
    print("No lower-unit-price alternatives match, the category and ingredient requirements.")
    
# NaN means the corresponding comparison was unavailable

,product_id,product_name,brand_name,size,price_usd,normalized_highlights
0,P439055,GENIUS Sleeping Collagen Moisturizer,Algenist,2 oz/ 60 mL,98.0,"[vegan, good_for:_loss_of_firmness, collagen, ..."


,product_id,product_name,brand_name,size,price_usd,ingredient_similarity,highlight_similarity,texture_similarity,combined_similarity,package_savings_usd,package_savings_pct,shared_highlights,shared_ingredients,highlights_used,texture_used
0,P453818,GENIUS Collagen Calming Relief,Algenist,1.35 oz/ 40 mL,58.0,0.223,0.802,NaN,0.351,40.0,40.8,"[collagen, hypoallergenic, skin_combination, s...","[caprylyl glycol, cetearyl alcohol, chlorella ...",True,False
1,P411403,Confidence in a Cream Anti-Aging Hydrating Moi...,IT Cosmetics,0.51 oz / 15 mL,20.0,0.114,0.750,NaN,0.255,78.0,79.6,"[good_for:_loss_of_firmness, skin_combination,...","[caprylic/capric triglyceride, caprylyl glycol...",True,False
2,P427641,Mini Confidence in a Cream Hydrating Moisturizer,IT Cosmetics,0.51 oz / 15 mL,20.0,0.114,0.750,NaN,0.255,78.0,79.6,"[good_for:_loss_of_firmness, skin_combination,...","[caprylic/capric triglyceride, caprylyl glycol...",True,False
3,P483698,Nourishing Squalane Daily Moisturizer,The Outset,1.7 oz / 50 mL,44.0,0.235,0.316,NaN,0.253,54.0,55.1,"[hypoallergenic, vegan]","[caprylic/capric triglyceride, caprylyl glycol...",True,False
4,P475185,24-7 Moisture Hydrating Day & Night Cream,TULA Skincare,1.5 oz/ 44 mL,54.0,0.117,0.535,NaN,0.210,44.0,44.9,"[good_for:_loss_of_firmness, skin_combination,...","[caprylic/capric triglyceride, citric acid, di...",True,False
